In [17]:
!pip install zipfile
!pip install pypdf
!pip install dotenv
!pip install pinecone
!pip install langchain_community
!pip install langchain_pinecone
!pip install langchain_openai
!pip install langchain_text_splitters

ERROR: Could not find a version that satisfies the requirement zipfile (from versions: none)
ERROR: No matching distribution found for zipfile
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pinecone
    Found existing installation: p

In [1]:
import os
from google.colab import userdata
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [2]:
import zipfile

ZIP_FILE_DIR = "./sample_data"
ZIP_FILE_NAME = "documents.zip"

with zipfile.ZipFile(ZIP_FILE_DIR + "/" + ZIP_FILE_NAME, 'r') as referencia_zip:
    referencia_zip.extractall(ZIP_FILE_DIR)

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./sample_data/documents/internal_docs_by_area/Customer_support/IronStore_Customer_Support_Escalation_Procedures.pdf")

pages = loader.load()

print(pages[0].page_content)

/tmp/ipykernel_3208/3731429197.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Customer Support Escalation Procedures
Owning area: Customer_support
Last reviewed: August 2026
Document ID: CS-POL-014
Version: 3.2
Applies to: IronStore customer support operations across Europe
1. Purpose and Overview
This procedure defines how IronStore identifies, manages, and escalates customer contacts that cannot be resolved
through standard frontline support. It is intended to ensure consistent decisions, timely ownership, and appropriate
protection of customer, payment, product, and company information.
Escalation is required when a case involves material customer impact, operational risk, legal or regulatory
considerations, reputational risk, or a resolution outside the agent’s authority. Escalation does not remove ownership
from the original agent unless a receiving team formally accepts the case.
The procedure applies to contacts received through email, telephone, chat, social media, marketplace messaging,
and the IronStore Help Centre.
2. Scope
This procedure covers:
  O

In [4]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

In [15]:
import re
from pathlib import Path
from langchain_core.documents import Document

def load_clean_document_content(file_name, file_path, pages):
    page_lines = []
    document_content = []

    for i, page in enumerate(pages):
        page_lines = [re.sub(r'\x7f|Page\s*\d+\s*of\s*\d+\s*—', "", line).strip() for line in page.page_content.split("\n") if line.strip()]
        if len(page_lines):
            document_content.append(
                Document(
                    page_content=" ".join(page_lines[1:]),
                    metadata={
                        "heading": page_lines[0],
                        "file_name": f"{file_name}_{i}",
                        "file_path": file_path
                    }
                )
              )
    return document_content

def get_documents_list_content(base_path):
    documents_list = []

    for file in base_path.rglob("*"):
        if file.is_file():
            file_name = file.name;
            if file_name != '.DS_Store':
                loader = PyPDFLoader(file.as_posix())
                pages = loader.load()
                file_path = file.relative_to(base_path).as_posix()
                documents_list.append(load_clean_document_content(file_name, file_path, pages))
            else:
                file.unlink()
    return documents_list

In [ ]:
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

PINECONE_INDEX_NAME = "enterprise-ai-assistant"
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# Create new pinecone index
if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536, # Standard dimensions for OpenAI embeddings
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

documents_list = get_documents_list_content(Path("./sample_data/documents"))

for document in documents_list:
  chunks = text_splitter.split_documents(document)

  if chunks:
      PineconeVectorStore.from_documents(
          documents=chunks,
          embedding=embeddings,
          index_name=PINECONE_INDEX_NAME,
          namespace=document[0].metadata.get("file_path", "unknown")
      )